# InsightForge — AI-Powered Business Intelligence Assistant

**Advanced Generative AI · Capstone**

This notebook walks through all seven steps of the problem statement against the
`insightforge` package that lives next to it.

| Step | Topic |
|---|---|
| 1 | Data preparation |
| 2 | Knowledge base creation |
| 3 | LLM application development (advanced summary + RAG integration) |
| 4 | Chain prompts |
| 5 | RAG system setup |
| 6 | Memory integration |
| 7 | LLMOps — QAEvalChain evaluation, visualisation, monitoring, Streamlit |

> Run `pip install -r ../requirements.txt` and set `OPENROUTER_API_KEY` first.
> Without a key everything still runs in offline extractive mode.

In [ ]:
import json
import sys
from pathlib import Path

# make the package importable when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd

from insightforge.config import settings

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

print("provider resolved to:", settings.resolve_provider())
print("chat model:", settings.chat_model)
print("dataset:", settings.data_file)

---
## Step 1 · Data preparation

The dataset is already clean, so this step is about *enrichment*: calendar
parts for time-series analysis, age bands and satisfaction bands for
segmentation.

In [ ]:
from insightforge.data_loader import dataset_profile, load_sales_data

df = load_sales_data()
print(df.shape)
df.head()

In [ ]:
print(json.dumps(dataset_profile(df), indent=2))

---
## Step 3a · Advanced data summary

`DataAnalyzer` produces the four families of metric the brief asks for:
sales by time period, product & regional analysis, customer segmentation, and
statistical measures.

In [ ]:
from insightforge.analysis import DataAnalyzer

analyzer = DataAnalyzer(df)
pd.Series(analyzer.kpis())

In [ ]:
# 1 · Sales performance by time period
analyzer.sales_by_period("Quarter").head(12)

In [ ]:
print(json.dumps(analyzer.growth_summary("Month"), indent=2))

In [ ]:
# 2 · Product and regional analysis
display(analyzer.product_performance())
display(analyzer.regional_performance())
analyzer.product_region_matrix()

In [ ]:
# 3 · Customer segmentation by demographics
segments = analyzer.customer_segments()
for name, table in segments.items():
    print(f"--- {name} ---")
    display(table)
analyzer.segment_matrix()

In [ ]:
# 4 · Statistical measures
print(json.dumps(analyzer.statistics(), indent=2))

---
## Step 2 · Knowledge base creation

Two knowledge sources are combined:

* the **metrics above**, rendered as short factual documents, and
* the **reference PDFs** in `Datasets/PDF Folder`, chunked with
  `RecursiveCharacterTextSplitter`.

Both go into a FAISS index. Because OpenRouter serves chat completions only,
embeddings are produced locally by a TF-IDF + LSA model implementing the
LangChain `Embeddings` interface.

In [ ]:
from insightforge.knowledge_base import KnowledgeBase

kb = KnowledgeBase(df, include_pdfs=True)
kb.build_vector_store()
kb.describe()

In [ ]:
# what a metric document actually looks like
print(kb.stat_documents[0].page_content)
print("\n---\n")
print(kb.stats_by_topic(["regional_performance"])[0].page_content)

---
## Step 3b · The custom retriever

Vector search alone cannot answer *"how did Widget B do in the West in 2024?"* —
no pre-computed document holds that cell. `BusinessStatsRetriever` routes the
question by keyword, slices the dataframe with pandas at query time, and only
then back-fills from the vector index.

In [ ]:
from insightforge.retriever import BusinessStatsRetriever, format_documents, route_topics

retriever = BusinessStatsRetriever(knowledge_base=kb, k=5)

question = "How did Widget B do in the West in 2024?"
print("routed topics:", route_topics(question))
docs = retriever.invoke(question)
print(f"{len(docs)} documents retrieved\n")
print(docs[0].page_content)

---
## Steps 4, 5 & 6 · RAG chain, chained prompts and memory

`InsightForgeAssistant` wires the retriever, the prompt library, the chat model,
a sliding-window memory and the monitoring log into one object.

In [ ]:
from insightforge.rag_chain import InsightForgeAssistant

assistant = InsightForgeAssistant(kb)
print("provider:", assistant.provider)

result = assistant.ask("Which product generates the most revenue and what share of the total is that?")
print(result["answer"])
print("\n[", result["provider"], "|", result["latency_s"], "s |", len(result["sources"]), "sources ]")

In [ ]:
# Memory in action: the follow-up has no subject of its own
follow_up = assistant.ask("And how does it do in the West region?")
print("condensed to:", follow_up["standalone_question"])
print()
print(follow_up["answer"])

In [ ]:
# what the model was actually given
for source in result["sources"][:3]:
    print(f"[{source['source']} · {source['topic']}]")
    print(source["excerpt"][:300], "\n")

### Step 4 · Chained prompts

`full_report()` runs three prompts in sequence, each consuming the previous
output: **metrics → insights → recommendations → executive summary**.

In [ ]:
report = assistant.full_report(n_insights=5)
print(report["executive_summary"])

In [ ]:
print(report["insights"])

In [ ]:
print(report["recommendations"])

---
## Step 7a · Model evaluation with QAEvalChain

The ground truth is computed from the dataset itself, so it can never drift.
Each answer is graded twice: by LangChain's `QAEvalChain` (LLM-as-judge) and by
a deterministic numeric-grounding check.

In [ ]:
from insightforge.evaluation import build_eval_set, evaluate, save_report

examples = build_eval_set(kb)
pd.DataFrame(examples).head()

In [ ]:
evaluation = evaluate(assistant, examples)
print(json.dumps(evaluation["metrics"], indent=2))
evaluation["table"][["question", "qaeval_grade", "numerically_grounded", "latency_s"]]

In [ ]:
save_report(evaluation)

---
## Step 7b · Data visualisation

Four families of chart: sales trends over time, product performance
comparisons, regional analysis, and customer demographics/segmentation.

In [ ]:
%matplotlib inline
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

from insightforge.visualization import (
    plot_correlation_matrix,
    plot_customer_demographics,
    plot_product_performance,
    plot_product_trend,
    plot_regional_analysis,
    plot_sales_trend,
    plot_seasonality,
    plot_segment_heatmap,
)

plot_sales_trend(df, "Month");

In [ ]:
plot_seasonality(df);

In [ ]:
plot_product_performance(df);
plot_product_trend(df);

In [ ]:
plot_regional_analysis(df);

In [ ]:
plot_customer_demographics(df);
plot_segment_heatmap(df);
plot_correlation_matrix(df);

---
## Step 7c · Monitoring

Every call is appended to `outputs/interactions.jsonl` with latency, tokens,
provider and errors — the telemetry behind the app's Monitoring tab.

In [ ]:
from insightforge.monitoring import load_interactions, usage_report

print(json.dumps(usage_report(), indent=2))
pd.DataFrame(load_interactions()).tail(8)[["timestamp", "question", "provider", "latency_s", "total_tokens"]]

---
## Step 7d · Streamlit UI

The interface lives in `streamlit_app.py` at the project root (Streamlit
Community Cloud looks for that name by default). From a terminal in the
project folder:

```bash
streamlit run streamlit_app.py
```

It exposes seven tabs — Chat, Dashboard, Insights, Evaluation, Monitoring and
Knowledge base — over exactly the objects built in this notebook.

---

### Summary

| Step | Delivered by |
|---|---|
| 1 Data preparation | `insightforge/data_loader.py` |
| 2 Knowledge base | `insightforge/knowledge_base.py` |
| 3 Advanced summary + custom retriever | `analysis.py`, `retriever.py` |
| 4 Chain prompts | `prompts.py`, `rag_chain.full_report()` |
| 5 RAG system | `rag_chain.py` (LCEL over the custom retriever) |
| 6 Memory | `rag_chain.WindowedMemory` + question condensation |
| 7 LLMOps | `evaluation.py`, `visualization.py`, `monitoring.py`, `streamlit_app.py` |